## Data Quality Assessment & Pipeline Code: Loan Approval Applications (Nested JSON to Modeling-Ready Table)

#### 0. Setup & Data Load

In [1]:
import pandas as pd 
import numpy as np
import ast
import json
import re

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
import joblib

df = pd.read_json("../data/raw_credit_applications.json")



#### 1. Raw Data Inspection
Purpose: confirm schema, types, and missingness before transforming anything

In [2]:
print("Raw shape:", df.shape)
display(df.head(3))

Raw shape: (502, 8)


,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN


In [3]:
# Schema and missingness snapshot
display(df.info())
missing_pct_raw = (df.isna().mean().sort_values(ascending=False) * 100).round(1)
display(missing_pct_raw.to_frame("missing_% (raw)"))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   _id                   502 non-null    object
 1   applicant_info        502 non-null    object
 2   financials            502 non-null    object
 3   spending_behavior     502 non-null    object
 4   decision              502 non-null    object
 5   processing_timestamp  62 non-null     object
 6   loan_purpose          50 non-null     object
 7   notes                 2 non-null      object
dtypes: object(8)
memory usage: 31.5+ KB


None

,missing_% (raw)
notes,99.6
loan_purpose,90.0
processing_timestamp,87.6
_id,0.0
spending_behavior,0.0
financials,0.0
applicant_info,0.0
decision,0.0


#### 2. Helper Functions
Reusable utilities for parsing, boolean standardization, and spending feature extraction


In [4]:
# a) SAFE PARSE (handles both dicts and stringified dicts) 
NULL_LIKE = {"", " ", "na", "n/a", "nan", "none", "null", "nil", "undefined"}

def safe_parse(val):
    """
    Parse dict/list if already native, or parse JSON / python-literal strings.
    Returns {} when parsing fails (best for dict-like columns).
    """
    if val is None:
        return {}
    if isinstance(val, (dict, list)):
        return val
    if isinstance(val, float) and pd.isna(val):
        return {}
    if isinstance(val, str):
        s = val.strip()
        if not s or s.lower() in NULL_LIKE:
            return {}
        # Handles true/false/null
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            pass
        try:
            return ast.literal_eval(s)
        except (ValueError, SyntaxError):
            return {}
    return {}


In [5]:
# b) TO_BOOL
def to_bool(x):
    if isinstance(x, bool):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, float)):
        return bool(x)
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"true", "1", "yes", "y"}:
            return True
        if s in {"false", "0", "no", "n"}:
            return False
    return np.nan


In [6]:
def clean_cat(cat):
    cat = (cat or "unknown").strip().lower()
    cat = re.sub(r"\s+", "_", cat)
    cat = re.sub(r"[^a-z0-9_]", "", cat)
    return cat or "unknown"


#### 3. Flatten Nested Structures (applicant_info, financials, decision)
Normalize nested JSON into a flat tabular structure.
Keep only the label from "decision" to prevent leakage


In [7]:
# Flatten applicant_info 
if "applicant_info" not in df.columns:
    raise ValueError("Column 'applicant_info' not found in dataframe.")

applicant_parsed = df["applicant_info"].apply(safe_parse)
applicant_df = pd.json_normalize(applicant_parsed)
applicant_df.columns = ["applicant_" + c for c in applicant_df.columns]

# Flatten financials 
if "financials" not in df.columns:
    raise ValueError("Column 'financials' not found in dataframe.")

financials_parsed = df["financials"].apply(safe_parse)
financials_df = pd.json_normalize(financials_parsed)
financials_df.columns = ["fin_" + c for c in financials_df.columns]

# Flatten decision (Leakage-safe) 
if "decision" not in df.columns:
    raise ValueError("Column 'decision' not found in dataframe.")

decision_parsed = df["decision"].apply(safe_parse)
decision_df = pd.json_normalize(decision_parsed)
decision_df.columns = ["decision_" + c for c in decision_df.columns]

# Ensure target exists
if "decision_loan_approved" not in decision_df.columns:
    raise ValueError(
        "Column 'decision_loan_approved' not found after flattening decision."
    )

# Keep only the target to prevent leakage
decision_df = decision_df[["decision_loan_approved"]].copy()

print("Flattening complete.")
print("Applicant columns:", applicant_df.columns.tolist())
print("Financial columns:", financials_df.columns.tolist())
print("Decision columns kept:", decision_df.columns.tolist())


Flattening complete.
Applicant columns: ['applicant_full_name', 'applicant_email', 'applicant_ssn', 'applicant_ip_address', 'applicant_gender', 'applicant_date_of_birth', 'applicant_zip_code']
Financial columns: ['fin_annual_income', 'fin_credit_history_months', 'fin_debt_to_income', 'fin_savings_balance', 'fin_annual_salary']
Decision columns kept: ['decision_loan_approved']


#### 4. Spending Behaviour Feature Engineering
Convert transaction lists into structured features (category totals + aggregate behaviour metrics)


In [8]:
def parse_spending(val):
    """
    Convert spending_behavior into:
    - spend_<category> totals
    - aggregate behavior features (txn count, total, mean, max, unique cats)
    """
    # Missing cases
    if val is None:
        return {}
    if isinstance(val, float) and pd.isna(val):
        return {}

    # Ensure list
    items = val if isinstance(val, list) else safe_parse(val)
    if not isinstance(items, list):
        return {}

    out = {}
    amounts = []
    cats_seen = set()

    for item in items:
        if not isinstance(item, dict):
            continue

        cat = clean_cat(item.get("category"))
        amt = item.get("amount", 0)

        try:
            amt = float(amt)
        except (TypeError, ValueError):
            amt = 0.0

        out[f"spend_{cat}"] = out.get(f"spend_{cat}", 0.0) + amt
        amounts.append(amt)
        cats_seen.add(cat)

    # Aggregate spend features (high value)
    out["spend_txn_count"] = int(len(amounts))
    out["spend_total"] = float(sum(amounts))
    out["spend_mean"] = float(np.mean(amounts)) if amounts else 0.0
    out["spend_max"] = float(np.max(amounts)) if amounts else 0.0
    out["spend_unique_cats"] = int(len(cats_seen))

    return out

spending_parsed = df["spending_behavior"].apply(parse_spending)
spending_df = pd.DataFrame(list(spending_parsed)).fillna(0)

print("spending_df shape:", spending_df.shape)
display(spending_df.head(3))



spending_df shape: (502, 20)


,spend_shopping,spend_rent,spend_alcohol,spend_txn_count,spend_total,spend_mean,spend_max,spend_unique_cats,spend_dining,spend_healthcare,spend_fitness,spend_entertainment,spend_insurance,spend_travel,spend_transportation,spend_utilities,spend_groceries,spend_education,spend_adult_entertainment,spend_gambling
0,480.0,790.0,247.0,3,1517.0,505.666667,790.0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,608.0,0.0,3,947.0,315.666667,608.0,3,96.0,243.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,109.0,0.0,1,109.0,109.000000,109.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### 5. Combine into a Modelling-Ready Clean Dataset
Join all extracted feature blocks into a single modelling-ready dataframe


In [9]:
# Combine Everything

base_cols = [c for c in ["_id", "processing_timestamp", "loan_purpose", "notes"] if c in df.columns]

df_clean = pd.concat(
    [
        df[base_cols].reset_index(drop=True),
        applicant_df.reset_index(drop=True),
        financials_df.reset_index(drop=True),
        decision_df.reset_index(drop=True),
        spending_df.reset_index(drop=True),
    ],
    axis=1,
)


#### 6. Data Quality Assessment
Quantify issues across completness, consistency, validity, and accuracy.


In [10]:
# 1) Completeness: missingness (count + %) 
missing_count = df_clean.isna().sum().sort_values(ascending=False)
missing_pct = (df_clean.isna().mean().sort_values(ascending=False) * 100).round(1)
dq_missing = pd.concat([missing_count.rename("missing_count"), missing_pct.rename("missing_%")], axis=1)
display(dq_missing.head(20))

# Drop extremely sparse columns (documented)
to_drop_sparse = []
if "notes" in df_clean.columns:
    to_drop_sparse.append("notes")

# Example: drop fin_annual_salary if present (very sparse / redundant vs fin_annual_income)
if "fin_annual_salary" in df_clean.columns:
    to_drop_sparse.append("fin_annual_salary")

df_clean.drop(columns=to_drop_sparse, inplace=True, errors="ignore")
print("Dropped sparse columns:", to_drop_sparse)


,missing_count,missing_%
notes,500,99.6
fin_annual_salary,497,99.0
loan_purpose,452,90.0
processing_timestamp,440,87.6
applicant_ssn,5,1.0
fin_annual_income,5,1.0
applicant_ip_address,5,1.0
applicant_zip_code,1,0.2
applicant_date_of_birth,1,0.2
applicant_gender,1,0.2


Dropped sparse columns: ['notes', 'fin_annual_salary']


In [11]:
# 2) Consistency: standardize label + basic category normalization checks 
df_clean["decision_loan_approved"] = df_clean["decision_loan_approved"].apply(to_bool)
invalid_label_n = df_clean["decision_loan_approved"].isna().sum()
print("Invalid/unrecognized label values (count):", int(invalid_label_n))

# Drop invalid labels (can't model/evaluate on them)
df_clean = df_clean.dropna(subset=["decision_loan_approved"]).copy()
df_clean["decision_loan_approved"] = df_clean["decision_loan_approved"].astype(bool)

print("Label distribution:")
display(df_clean["decision_loan_approved"].value_counts(dropna=False).to_frame("count"))

# Example categorical consistency check (if gender exists)
if "applicant_gender" in df_clean.columns:
    raw_gender = df_clean["applicant_gender"].astype(str).str.strip()
    top_gender = raw_gender.value_counts(dropna=False).head(15)
    print("Top gender raw values (showing possible inconsistencies):")
    display(top_gender.to_frame("count"))

    # Normalize a few common variants 
gender_norm = raw_gender.str.strip().str.lower()

gender_norm = gender_norm.replace({
    "m": "Male",
    "male": "Male",
    "man": "Male",
    "f": "Female",
    "female": "Female",
    "woman": "Female",
    "": np.nan,
    "nan": np.nan,
    "none": np.nan,
})

df_clean["applicant_gender"] = gender_norm

# Verify normalization
print("Gender values after normalization:")
display(
    df_clean["applicant_gender"]
    .value_counts(dropna=False)
    .to_frame("count")
)


Invalid/unrecognized label values (count): 0
Label distribution:


,count
decision_loan_approved,
True,292
False,210


Top gender raw values (showing possible inconsistencies):


,count
applicant_gender,
Male,195
Female,193
F,58
M,53
,2
nan,1


Gender values after normalization:


,count
applicant_gender,
Female,251
Male,248
NaN,3


In [12]:
# 3) Validity: numeric type coercion + impossible values 
num_cols = [c for c in df_clean.columns if c.startswith(("fin_", "spend_"))]
for c in num_cols:
    df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")

# Basic validity rules (counts before remediation)
validity_issues = {}

if "fin_annual_income" in df_clean.columns:
    validity_issues["fin_annual_income_negative_or_zero"] = int((df_clean["fin_annual_income"] <= 0).sum())
if "fin_credit_history_months" in df_clean.columns:
    validity_issues["fin_credit_history_months_negative"] = int((df_clean["fin_credit_history_months"] < 0).sum())
if "spend_total" in df_clean.columns:
    validity_issues["spend_total_negative"] = int((df_clean["spend_total"] < 0).sum())

display(pd.Series(validity_issues, name="issue_count").to_frame())

# Remediation: set invalid values to NaN (so they can be imputed later in the ML pipeline)
if "fin_annual_income" in df_clean.columns:
    df_clean.loc[df_clean["fin_annual_income"] <= 0, "fin_annual_income"] = np.nan
if "fin_credit_history_months" in df_clean.columns:
    df_clean.loc[df_clean["fin_credit_history_months"] < 0, "fin_credit_history_months"] = np.nan
if "spend_total" in df_clean.columns:
    df_clean.loc[df_clean["spend_total"] < 0, "spend_total"] = np.nan


,issue_count
fin_annual_income_negative_or_zero,1
fin_credit_history_months_negative,2
spend_total_negative,0


In [13]:
# 4) Accuracy (reasonableness): simple anomaly scans 
# These are heuristics (we can't fully verify "truth" without an external source).

checks = {}

# Income vs spending sanity (if both exist)
if "fin_annual_income" in df_clean.columns and "spend_total" in df_clean.columns:
    # spending greater than income can be real, but flag extreme cases for review
    ratio = df_clean["spend_total"] / (df_clean["fin_annual_income"] + 1e-6)
    checks["spend_to_income_ratio_gt_2"] = int((ratio > 2).sum())
    checks["spend_to_income_ratio_gt_5"] = int((ratio > 5).sum())

display(pd.Series(checks, name="flag_count").to_frame())


,flag_count
spend_to_income_ratio_gt_2,0
spend_to_income_ratio_gt_5,0


#### 7. Remove Direct Identifiers (Modelling Dataset)
Remove direct identifiers from the modelling table


In [14]:
# Personal data needs to be removed
df_clean.drop(
    columns=[
        "applicant_full_name",
        "applicant_email",
        "applicant_phone",
        "applicant_address",
    ],
    inplace=True,
    errors="ignore",
)


#### 8. Optional Leakage-Risk Fiels
Assess and drop fields that are likely post-decidion / workflow-derived


In [15]:
# If present, assess sparsity by label (as a proxy for workflow-derived fields)
if "processing_timestamp" in df_clean.columns:
    print("processing_timestamp availability by label (proportion):")
    display(df_clean.groupby("decision_loan_approved")["processing_timestamp"].apply(lambda s: s.notna().mean()).to_frame("not_null_rate"))

if "loan_purpose" in df_clean.columns:
    print("loan_purpose availability by label (proportion):")
    display(df_clean.groupby("decision_loan_approved")["loan_purpose"].apply(lambda s: s.notna().mean()).to_frame("not_null_rate"))

# Drop to avoid leakage risk and because of extreme sparsity in this dataset
df_clean.drop(columns=["processing_timestamp", "loan_purpose"], inplace=True, errors="ignore")


processing_timestamp availability by label (proportion):


,not_null_rate
decision_loan_approved,
False,0.104762
True,0.136986


loan_purpose availability by label (proportion):


,not_null_rate
decision_loan_approved,
False,0.114286
True,0.089041


#### 9. Duplicate Checks
Check for duplicates at multiple levels: full rows, _id, and a fingerprint of key features


In [16]:
print("Full row duplicates:", int(df_clean.duplicated().sum()))

if "_id" in df_clean.columns:
    print("Duplicate _ids:", int(df_clean["_id"].duplicated().sum()))
    dupes = df_clean[df_clean["_id"].duplicated(keep=False)]
    cols = [c for c in ["_id", "decision_loan_approved", "fin_annual_income", "fin_credit_history_months"] if c in df_clean.columns]
    display(dupes[cols].head(20))

# Fingerprint duplicates (after PII removal) — heuristic for "same applicant"
fingerprint_cols = [c for c in [
    "fin_annual_income",
    "fin_credit_history_months",
    "applicant_gender",
    "spend_total",
    "spend_txn_count",
    "spend_unique_cats",
] if c in df_clean.columns]

fp = df_clean.copy()
for c in fingerprint_cols:
    fp[c] = fp[c].fillna("__MISSING__")

fp_dupes = fp.duplicated(subset=fingerprint_cols, keep=False)
print("Fingerprint duplicates (count):", int(fp_dupes.sum()))
if fp_dupes.any():
    display(df_clean.loc[fp_dupes, ["_id"] + fingerprint_cols].sort_values(fingerprint_cols))



Full row duplicates: 1
Duplicate _ids: 2


,_id,decision_loan_approved,fin_annual_income,fin_credit_history_months
8,app_042,False,69000.0,43.0
354,app_042,False,69000.0,43.0
383,app_001,False,102000.0,37.0
455,app_001,False,102000.0,37.0


Fingerprint duplicates (count): 2


,_id,fin_annual_income,fin_credit_history_months,applicant_gender,spend_total,spend_txn_count,spend_unique_cats
8,app_042,69000.0,43.0,Male,621.0,2,2
354,app_042,69000.0,43.0,Male,621.0,2,2


#### 10. Final Validity Checks
Prove the dataset is modelling-ready and leakage-safe


In [17]:
print("Final shape:", df_clean.shape)

# 1) Label check
assert "decision_loan_approved" in df_clean.columns, "Missing target column!"
assert df_clean["decision_loan_approved"].dropna().isin([True, False]).all(), "Target is not clean boolean!"
print("Target looks clean.")

# 2) Leakage check
leak_cols = [c for c in df_clean.columns if c.startswith("decision_") and c != "decision_loan_approved"]
assert len(leak_cols) == 0, f"Leakage columns still present: {leak_cols}"
print("No decision leakage columns present.")

# 3) No nested leftovers
nested_left = [c for c in ["applicant_info", "financials", "decision", "spending_behavior"] if c in df_clean.columns]
assert len(nested_left) == 0, f"Nested raw columns still present: {nested_left}"
print("No raw nested columns left.")

# 4) Numeric sanity checks
num_cols = [c for c in df_clean.columns if c.startswith(("fin_", "spend_"))]
bad_numeric = df_clean[num_cols].select_dtypes(exclude=["number"]).columns.tolist()
assert len(bad_numeric) == 0, f"These fin_/spend_ columns are not numeric: {bad_numeric}"
print("fin_ and spend_ columns are numeric.")

# 5) Final missingness snapshot
missing_pct = (df_clean.isna().mean().sort_values(ascending=False) * 100).round(1)
display(missing_pct.head(20).to_frame("missing_%"))


Final shape: (502, 31)
Target looks clean.
No decision leakage columns present.
No raw nested columns left.
fin_ and spend_ columns are numeric.


,missing_%
fin_annual_income,1.2
applicant_ip_address,1.0
applicant_ssn,1.0
applicant_gender,0.6
fin_credit_history_months,0.4
applicant_zip_code,0.2
applicant_date_of_birth,0.2
_id,0.0
fin_debt_to_income,0.0
fin_savings_balance,0.0


#### 11. Pipeline Code

In [18]:
# FINAL MODELING PIPELINE (sklearn)
# Target: decision_loan_approved

# 1) X / y
TARGET = "decision_loan_approved"
if TARGET not in df_clean.columns:
    raise ValueError(f"Missing target column: {TARGET}")

# y as 0/1
y = df_clean[TARGET].astype(int)

# X = everything except target (+ drop _id if present)
X = df_clean.drop(columns=[TARGET], errors="ignore").copy()
if "_id" in X.columns:
    X = X.drop(columns=["_id"])

# 2) Column lists
numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("Rows:", len(df_clean), "| Num features:", len(numeric_cols), "| Cat features:", len(categorical_cols))

# 3) Preprocessing
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  # helps Logistic Regression
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ],
    remainder="drop"
)

# 4) Model + full pipeline
model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    solver="lbfgs"
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# 5) Cross-validation 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc")
print("5-fold CV ROC-AUC:", cv_auc.mean().round(4), "+/-", cv_auc.std().round(4))

# 6) Holdout test split (useful as a final check)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("\n=== Holdout Test Results ===")
print("ROC-AUC:", roc_auc_score(y_test, y_proba).round(4))
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# 7) Save the trained pipeline
joblib.dump(clf, "loan_approval_pipeline.joblib")
print("\nSaved: loan_approval_pipeline.joblib")


Rows: 502 | Num features: 24 | Cat features: 5
5-fold CV ROC-AUC: 0.6025 +/- 0.0128

=== Holdout Test Results ===


AttributeError: 'float' object has no attribute 'round'